# Templisafe - library quickstart

## Minimal working example

### Template

In [1]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import SourceSettings

sql_template_content: str = """SELECT 
{%- for col in select_columns %}
  {{ col }}{% if not loop.last %},{% endif %}
{%- endfor %}
FROM users u
  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}
WHERE TRUE 
  AND u.code IN ({{ user_codes | join(', ') }})
  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}
  AND u.status = '{{ user_status }}'
  AND m.code = '{{ metric_code }}'
  AND m.is_updated IS {{ metric_updated_flag }}
  AND m.threshold > {{ metric_threshold }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content=sql_template_content, 
    content_type=ContentType.TEXT
)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n{%- for col in select_columns %}\n  {{ col }}{% if not loop.last %},{% endif %}\n{%- endfor %}\nFROM users u\n  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}\nWHERE TRUE \n  AND u.code IN ({{ user_codes | join(', ') }})\n  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}\n  AND u.status = '{{ user_status }}'\n  AND m.code = '{{ metric_code }}'\n  AND m.is_updated IS {{ metric_updated_flag }}\n  AND m.threshold > {{ metric_threshold }}\n")

### Schema

In [2]:
schema_content: str = """
schema:
  select_columns:
    type: list
    default:
      - u.id
      - u.name
      - u.code
      - u.age
      - u.status
      - m.is_updated
      - m.threshold
    metadata:
      description: Target select list
      title: SELECT LIST

  user_join_key: 
    type: str
    metadata:
      description: Join key for table user

  metric_join_key: str
    
  user_codes: 
    type: list
    default:
      - 10
      - 11
      - 12
    
  user_age_lower:
    type: int
    default: 0
    constraints:
      ge: 0

  user_age_upper:
    type: int
    default: 1000000
    constraints:
      ge: 0

  user_status: str

  metric_code:
    type: str
    default: "9999910"
    constraints:
      max_length: 12

  metric_updated_flag:
    type: bool
    default: true

  metric_threshold: float
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type=ContentType.YAML, 
    content=schema_content
 )
schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nschema:\n  select_columns:\n    type: list\n    default:\n      - u.id\n      - u.name\n      - u.code\n      - u.age\n      - u.status\n      - m.is_updated\n      - m.threshold\n    metadata:\n      description: Target select list\n      title: SELECT LIST\n\n  user_join_key: \n    type: str\n    metadata:\n      description: Join key for table user\n\n  metric_join_key: str\n\n  user_codes: \n    type: list\n    default:\n      - 10\n      - 11\n      - 12\n\n  user_age_lower:\n    type: int\n    default: 0\n    constraints:\n      ge: 0\n\n  user_age_upper:\n    type: int\n    default: 1000000\n    constraints:\n      ge: 0\n\n  user_status: str\n\n  metric_code:\n    type: str\n    default: "9999910"\n    constraints:\n      max_length: 12\n\n  metric_updated_flag:\n    type: bool\n    default: true\n\n  metric_threshold: float\n')

### Variants

In [3]:
variants_content: str = """
variants:
  select_columns:
    - u.id
    - u.name
    - u.age
    - m.value

  user_join_key: id
  metric_join_key: user_id
  # user_codes -> defaulted
    
  user_age_lower:  18
  user_age_upper: 70
  user_status: ACTIVE

  # metric_code -> defaulted
  metric_updated_flag: true
  metric_threshold: 54.12
"""
variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type=ContentType.YAML, 
    content=variants_content
)
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nvariants:\n  select_columns:\n    - u.id\n    - u.name\n    - u.age\n    - m.value\n\n  user_join_key: id\n  metric_join_key: user_id\n  # user_codes -> defaulted\n\n  user_age_lower:  18\n  user_age_upper: 70\n  user_status: ACTIVE\n\n  # metric_code -> defaulted\n  metric_updated_flag: true\n  metric_threshold: 54.12\n')

### Compilation

#### Create the `Templater`

In [4]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()

templater: Templater = factory.create()
templater

#### Compile

In [5]:
from templisafe.templater import Compilation

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

<Outcome.SUCCESS: 0>

In [6]:
compilation

Compilation(outcome=<Outcome.SUCCESS: 0>, message='Query successfully compiled with schema', diagnostics=(), _spec=CompilationSpec(template=Template(template_str="SELECT \n{%- for col in select_columns %}\n  {{ col }}{% if not loop.last %},{% endif %}\n{%- endfor %}\nFROM users u\n  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}\nWHERE TRUE \n  AND u.code IN ({{ user_codes | join(', ') }})\n  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}\n  AND u.status = '{{ user_status }}'\n  AND m.code = '{{ metric_code }}'\n  AND m.is_updated IS {{ metric_updated_flag }}\n  AND m.threshold > {{ metric_threshold }}\n", vars={'user_join_key', 'metric_join_key', 'user_status', 'user_age_upper', 'metric_threshold', 'metric_code', 'user_codes', 'user_age_lower', 'select_columns', 'metric_updated_flag'}), schema=Schema(model_cls=<class 'abc.ModelSchema'>)))

In [7]:
compilation.compiled.schema.model_cls.model_fields

{'select_columns': FieldInfo(annotation=list, required=False, default=['u.id', 'u.name', 'u.code', 'u.age', 'u.status', 'm.is_updated', 'm.threshold'], title='SELECT LIST', description='Target select list', json_schema_extra={'_index': 0}),
 'user_join_key': FieldInfo(annotation=str, required=True, description='Join key for table user', json_schema_extra={'_index': 1}),
 'metric_join_key': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 2}),
 'user_codes': FieldInfo(annotation=list, required=False, default=[10, 11, 12], json_schema_extra={'_index': 3}),
 'user_age_lower': FieldInfo(annotation=int, required=False, default=0, json_schema_extra={'_index': 4}, metadata=[Ge(ge=0)]),
 'user_age_upper': FieldInfo(annotation=int, required=False, default=1000000, json_schema_extra={'_index': 5}, metadata=[Ge(ge=0)]),
 'user_status': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 6}),
 'metric_code': FieldInfo(annotation=str, required=False, default='

#### Compile without a `Schema`

In [8]:
compilation_empty_schema: Compilation = templater.compile(template_source=template_inline_source_settings)      # No schema provided
compilation_empty_schema.outcome

<Outcome.SUCCESS: 0>

In [9]:
compilation_empty_schema.message

'Query successfully compiled with empty schema'

In [10]:
from templisafe.template.template_model import CompilationSpec, Schema

compiled_empty_schema: CompilationSpec = compilation_empty_schema.compiled
empty_schema: Schema = compiled_empty_schema.schema
empty_schema.model_cls.model_fields

{'user_join_key': FieldInfo(annotation=object, required=False, default=None),
 'metric_join_key': FieldInfo(annotation=object, required=False, default=None),
 'user_status': FieldInfo(annotation=object, required=False, default=None),
 'user_age_upper': FieldInfo(annotation=object, required=False, default=None),
 'metric_threshold': FieldInfo(annotation=object, required=False, default=None),
 'metric_code': FieldInfo(annotation=object, required=False, default=None),
 'user_codes': FieldInfo(annotation=object, required=False, default=None),
 'user_age_lower': FieldInfo(annotation=object, required=False, default=None),
 'select_columns': FieldInfo(annotation=object, required=False, default=None),
 'metric_updated_flag': FieldInfo(annotation=object, required=False, default=None)}

### Rendering

In [11]:
from templisafe.settings.source_settings import InlineSourceSettings

assert isinstance(variants_inline_source_settings, InlineSourceSettings)
print(variants_inline_source_settings.content)


variants:
  select_columns:
    - u.id
    - u.name
    - u.age
    - m.value

  user_join_key: id
  metric_join_key: user_id
  # user_codes -> defaulted

  user_age_lower:  18
  user_age_upper: 70
  user_status: ACTIVE

  # metric_code -> defaulted
  metric_updated_flag: true
  metric_threshold: 54.12



In [12]:
from templisafe.templater import Rendering

rendering: Rendering = templater.render(
    compiled=compilation.compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

<Outcome.SUCCESS: 0>

In [13]:
from templisafe.template.template_model import RenderingSpec

rendered: RenderingSpec = rendering.rendered 
rendered.names

{'default_1'}

In [14]:
for r in rendered.parameterizations: 
    print(r.rendered_str)

SELECT
  u.id,
  u.name,
  u.age,
  m.value
FROM users u
  JOIN metrics m ON u.id = m.user_id
WHERE TRUE 
  AND u.code IN (10, 11, 12)
  AND u.age BETWEEN 18 AND 70
  AND u.status = 'ACTIVE'
  AND m.code = '9999910'
  AND m.is_updated IS True
  AND m.threshold > 54.12


In [15]:
rendered.parameterizations[0].variant.names

{'metric_join_key',
 'metric_threshold',
 'metric_updated_flag',
 'select_columns',
 'user_age_lower',
 'user_age_upper',
 'user_join_key',
 'user_status'}

In [16]:
rendered.parameterizations[0].variant.bindings

[Binding(index=0, name='select_columns', value=['u.id', 'u.name', 'u.age', 'm.value']),
 Binding(index=1, name='user_join_key', value='id'),
 Binding(index=2, name='metric_join_key', value='user_id'),
 Binding(index=3, name='user_age_lower', value=18),
 Binding(index=4, name='user_age_upper', value=70),
 Binding(index=5, name='user_status', value='ACTIVE'),
 Binding(index=6, name='metric_updated_flag', value=True),
 Binding(index=7, name='metric_threshold', value=54.12)]

### Build

#### Configuration files

In [17]:
TEMPLATE_PATH: str = "./template.sql.j2"
with open(TEMPLATE_PATH) as f:
    print(f.read())

SELECT 
  {% for col in select_columns -%}
    {{ col }}{% if not loop.last %},{% endif %}
  {% endfor %}
FROM users u
  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}
WHERE TRUE 
  AND u.code IN (
    {% for code in user_codes -%}
      {{ code }}{% if not loop.last %},{% endif %}
    {% endfor %}
  )
  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}
  AND u.status = '{{ user_status }}'
  AND m.code = '{{ metric_code }}'
  AND m.is_updated IS {{ metric_updated_flag }}
  AND m.threshold > {{ metric_threshold }}


In [18]:
SCHEMA_PATH: str = "./schema.yaml"
with open(SCHEMA_PATH) as f:
    print(f.read())


schema:
  select_columns:
    type: list
    default:
      - u.id
      - u.name
      - u.code
      - u.age
      - u.status
      - m.is_updated
      - m.threshold

  user_join_key: str

  metric_join_key: str

  user_codes: 
    type: list
    default:
      - 10
      - 11
      - 12
    
  user_age_lower:
    type: int
    default: 0

  user_age_upper:
    type: int
    default: 1000000

  user_status: str

  metric_code:
    type: str
    default: "9999910"

  metric_updated_flag:
    type: bool
    default: true

  metric_threshold: float



In [19]:
VARIANTS_PATH: str = "./variants.yaml"
with open(VARIANTS_PATH) as f:
    print(f.read())

variants:
  all_ages:                       # variant 1
    select_columns:
      - u.id
      - u.name
      - u.age
      - m.value

    user_join_key: id
    metric_join_key: user_id
    # user_codes -> defaulted
      
    # user_age_lower -> defaulted
    # user_age_upper -> defaulted
    user_status: ACTIVE

    # metric_code -> defaulted
    metric_updated_flag: true
    metric_threshold: 54.12

  limited_ages:                   # variant 2
    select_columns:
      - u.id
      - u.name
      - u.age
      - m.value

    user_join_key: id
    metric_join_key: user_id
    # user_codes -> defaulted
      
    user_age_lower:  18
    user_age_upper: 70
    user_status: ACTIVE

    # metric_code -> defaulted
    metric_updated_flag: true
    metric_threshold: 54.12


#### Sources

Use a `LocalSource` to load configurations from a file:

In [20]:
template_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=TEMPLATE_PATH
)

schema_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=SCHEMA_PATH
)

variants_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH
)

template_local_source_settings, schema_local_source_settings, variants_local_source_settings

(LocalSourceSettings(content_type=None, path='./template.sql.j2'),
 LocalSourceSettings(content_type=None, path='./schema.yaml'),
 LocalSourceSettings(content_type=None, path='./variants.yaml'))

#### Build (compilation + rendering)

Use **build** to **compile** and **render** in one step:

In [21]:
from templisafe.templater import Build

build: Build = templater.build(
    template_source=template_local_source_settings,
    schema_source=schema_local_source_settings,
    variants_sources=variants_local_source_settings
)

build.outcome

<Outcome.SUCCESS: 0>

In [22]:
compilation: Compilation = build.compilation
compilation.outcome, compilation.message

(<Outcome.SUCCESS: 0>, 'Query successfully compiled with schema')

In [23]:
rendering: Rendering = build.rendering
rendering.outcome, rendering.message

(<Outcome.SUCCESS: 0>, 'Rendering successful')

In [24]:
for par_name, par in rendering.rendered.mapping.items(): 
    print("-" * 50)
    print(f"Variant '{par_name}':")
    print("-" * 50)
    print(par.rendered_str)

--------------------------------------------------
Variant 'all_ages':
--------------------------------------------------
SELECT 
  u.id,
  u.name,
  u.age,
  m.value
  
FROM users u
  JOIN metrics m ON u.id = m.user_id
WHERE TRUE 
  AND u.code IN (
    10,
    11,
    12
    
  )
  AND u.age BETWEEN 0 AND 1000000
  AND u.status = 'ACTIVE'
  AND m.code = '9999910'
  AND m.is_updated IS True
  AND m.threshold > 54.12
--------------------------------------------------
Variant 'limited_ages':
--------------------------------------------------
SELECT 
  u.id,
  u.name,
  u.age,
  m.value
  
FROM users u
  JOIN metrics m ON u.id = m.user_id
WHERE TRUE 
  AND u.code IN (
    10,
    11,
    12
    
  )
  AND u.age BETWEEN 18 AND 70
  AND u.status = 'ACTIVE'
  AND m.code = '9999910'
  AND m.is_updated IS True
  AND m.threshold > 54.12


## Query diagnostics

### Compilation

#### Undeclared variables

Template

In [25]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
  AND m.code = '{{ undeclared }}'
"""

template_inline_source_settings: InlineSourceSettings = InlineSourceSettings(content=sql_template_content, content_type=ContentType.TEXT)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n  AND m.code = '{{ undeclared }}'\n")

Schema

In [26]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

# Parameter 'undeclared' not declared in the schema
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=schema_content,
    content_type=ContentType.YAML
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n')

In [27]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create()

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

CompilationFailureError: Query compilation failed with outcome ERROR: Query compilation failed
Diagnostics:
[ERROR] variable=undeclared: Undeclared variable: 'undeclared'

#### Unused variables

Template

In [28]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=sql_template_content, 
    content_type=ContentType.TEXT
    )
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n")

Schema

In [29]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

# Parameter 'undeclared' not declared in the schema
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
  unused: any
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=schema_content,
    content_type=ContentType.YAML
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n  unused: any\n')

In [30]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create()

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

C:\p\prelios\templisafe\src\templisafe\templater.py:143: UserWarning: Query compiled with warnings
  self._handle_outcome(


<Outcome.WARNING: 1>

In [31]:
compilation.message, compilation.diagnostics

('Query successfully compiled with schema',
 (Diagnostic(level=<Outcome.WARNING: 1>, message="Unused variable: 'unused'", name='unused', index=None),))

### Rendering

Template

In [32]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=sql_template_content, 
    content_type=ContentType.TEXT
)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n")

Schema

In [33]:
from templisafe.util.util import ContentType

schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline",
    content=schema_content,
    content_type=ContentType.YAML
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n')

In [34]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create()

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

<Outcome.SUCCESS: 0>

In [35]:
from templisafe.template.template_model import CompilationSpec

compiled: CompilationSpec = compilation.compiled
compiled

CompilationSpec(template=Template(template_str="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n", vars={'col2', 'col1', 'user_status', 'user_age_lower'}), schema=Schema(model_cls=<class 'abc.ModelSchema'>))

In [36]:
compiled.schema.model_cls.model_fields

{'col1': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 0}),
 'col2': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 1}),
 'user_status': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 2}),
 'user_age_lower': FieldInfo(annotation=int, required=True, json_schema_extra={'_index': 3})}

#### Undeclared parameters

In [39]:
variants_content: str = """variants:
  col1: id
  col2: name
  user_status: ACTIVE
  user_age_lower: 18
  unused: unused 
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=variants_content, 
    content_type=ContentType.YAML
    )
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='variants:\n  col1: id\n  col2: name\n  user_status: ACTIVE\n  user_age_lower: 18\n  unused: unused \n')

In [40]:
from templisafe.template.template_model import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

C:\p\prelios\templisafe\src\templisafe\templater.py:181: UserWarning: Query rendered with warnings
  self._handle_outcome(


<Outcome.WARNING: 1>

In [41]:
rendering.message, rendering.diagnostics

('Rendering completed with warnings',
 (Diagnostic(level=<Outcome.WARNING: 1>, message="'default_1' - Extra binding provided: 'unused'", name='unused', index=4),))

#### Wrong parameter type

In [44]:
variants_content: str = """variants:
  col1: 5.67                      # Should be a string
  col2: name
  user_status: 1                  # Should be a string
  user_age_lower: [1, 2, 3]       # Should be an int
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=variants_content, 
    content_type=ContentType.YAML
    )
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='variants:\n  col1: 5.67                      # Should be a string\n  col2: name\n  user_status: 1                  # Should be a string\n  user_age_lower: [1, 2, 3]       # Should be an int\n')

In [45]:
from templisafe.template.template_model import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

RenderingError: Rendering(outcome=<Outcome.ERROR: 2>, message='Validation failed due to errors', diagnostics=(Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'col1': Input should be a valid string", name='col1', index=0), Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'user_status': Input should be a valid string", name='user_status', index=2), Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'user_age_lower': Input should be a valid integer", name='user_age_lower', index=3)), _spec=None)